# stage04 — Groups, symmetries, paths, loops, and homotopy

**Python route** · lab `stage04` · stage

Make your hand prediction first, then open the canonical worked notebook
(`notebooks/04_lab.ipynb`) and the [lesson](../../../lessons/04_lesson.md).
The canonical course is `lessons/` + references; this notebook adds an
interactive route through the native16 evidence, not a second curriculum.

## The native experiment for this lab

Rotate an observed digit and compare its pixels and closed-pixel Betti numbers.

## Boundary

A geometric invariance is not necessarily a class-label invariance. General homotopy proofs remain written assignments.

## Where every path points

This notebook is `workbooks/polyglot/python/stage04.ipynb`. Paths below are
defined once, here, so the same names mean the same thing in every lab.

| Path | What it is | Who owns it |
|---|---|---|
| `curriculum/catalog.json` | the single owner of the 31-stage lesson order | course |
| `native/protocols/labs.json` | the single owner of the 57 native experiment definitions | course |
| `lessons/` | canonical written explanations | course |
| `notebooks/` | original worked Python labs (**no** Go or Rust here) | course |
| `practice/` | original learner assignments | course |
| [`notebooks/04_lab.ipynb`](../../../notebooks/04_lab.ipynb) | this lab's canonical reference | course |
| `native/go/`, `native/rust/` | reference implementations the notebooks compile | read-only |
| `native/python/` | independent data and scientific checks | read-only |
| `native/guides/` | language and implementation guidance | read-only |
| [`native/guides/labs/stage04.md`](../../../native/guides/labs/stage04.md) | this lab's implementation guide | read-only |
| `tools/build_language_notebooks.py` | the generator that owns `workbooks/polyglot/` | tooling |
| `workbooks/polyglot/` | these generated teaching notebooks | generated |
| `my_work/` | your private attempts and evidence (git-ignored) | **you** |

### The one that matters most

| | |
|---|---|
| `native/python/` | the **reference**. Committed and shared. Do not edit. |
| `my_work/<session>/native/python/` | **your copy**, from `native_course.py prepare`. Edit here. |

### Relative to this notebook

| From here | Points at |
|---|---|
| [`../go/stage04.ipynb`](../go/stage04.ipynb) | the Go version of this lab |
| [`../README.md`](../README.md) | the index of all 57 labs, three languages |
| [`../../native/guides/LEARNING_PATHS.md`](../../../native/guides/LEARNING_PATHS.md) | the language learning paths |
| [`../../docs/release/LANGUAGE_NOTEBOOKS.md`](../../../docs/release/LANGUAGE_NOTEBOOKS.md) | how these notebooks work |

## 1. Setup

In [ ]:
"""Notebook setup. Run this first.

This reads the course catalogue and prepares a fresh output directory for this
lab. Nothing is written into the course tree and nothing is graded.
"""
import importlib.util, json, subprocess, sys
from pathlib import Path

# Locate the course root by walking up until native_course.py is found.
_here = Path.cwd().resolve()
ROOT = next(p for p in [_here, *_here.parents] if (p / "native_course.py").is_file())
sys.path.insert(0, str(ROOT / "src"))


def load_runner():
    spec = importlib.util.spec_from_file_location("shape_native_runner", ROOT / "native_course.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


runner = load_runner()
LAB_KEY = 'stage04'
LAB_ID = '04'
REFERENCE = 'notebooks/04_lab.ipynb'
LAB = next(r for r in runner.available_labs(ROOT) if r["key"] == LAB_KEY)
print("course root :", ROOT)
print("lab         :", LAB["key"], "-", LAB["title"])
print("kind        :", LAB["kind"])
print("reference   :", REFERENCE)
print("native goal :", LAB["native_goal"])
print("challenges  :", len(LAB["challenges"]))

## 2. Run the canonical Python experiment

In [ ]:
# --------------------------------------------------------------------------
# Run the CANONICAL Python experiment for this lab.
# --------------------------------------------------------------------------
# Reference: notebooks/04_lab.ipynb
# Kind     : notebook — the canonical worked notebook
#
# This is the original course material. It is deliberately separate from the
# native16 Go/Rust labs: those are matching teaching protocols, NOT
# operation-for-operation ports of this reference.

REF_OUT = ROOT / "my_work" / f"notebook_{LAB_KEY}_reference"
already = REF_OUT / "executed" / REFERENCE
if already.is_file():
    print("already executed :", already.relative_to(ROOT))
    print("delete that output directory to rerun from scratch.")
elif REF_OUT.exists():
    # A previous attempt left the directory without a finished run. Say so plainly
    # instead of letting run_references raise a bare FileExistsError.
    print("This lab's output directory already exists but holds no executed copy:")
    print(" ", REF_OUT.relative_to(ROOT))
    print()
    print("That is a leftover from an earlier attempt (possibly interrupted).")
    print("Evidence is never erased for you. Choose ONE of these:")
    print("  - keep the old attempt and run the reference somewhere new:")
    print(f"      uv run --no-sync python course.py run --stage {LAB_ID} --output my_work/stage{LAB_ID}_ref02")
    print("  - or, after inspecting it, move the old folder aside yourself.")
else:
    print("Running the canonical reference with a fresh kernel in an isolated")
    print("copied workspace. This is the REAL original experiment, not a")
    print("native16 re-implementation. It may take a minute.")
    sys.path.insert(0, str(ROOT / "src"))
    from shape_lab.course import run_references

    manifest = run_references(ROOT, [REFERENCE], REF_OUT, jobs=1, timeout=300)
    for record in manifest["records"]:
        print(f"{record.get('path')} -> {record.get('status')} {record.get('error', '')}")
    print()
    print("executed copy :", (REF_OUT / "executed" / REFERENCE).relative_to(ROOT))
    print("If it failed, read the error above and fix the cause. Do NOT replace")
    print("the canonical reference with a Python re-implementation of the lab.")

## 3. Which registered inputs does this lab consume? (input audit)

In [ ]:
# --------------------------------------------------------------------------
# Is an existing native run still valid? Source hashes must be unchanged.
# --------------------------------------------------------------------------
RUN_DIR = ROOT / "my_work" / "notebook_stage04_python"
RUN_DIR.mkdir(parents=True, exist_ok=True)

REGISTERED = [
    "data/raw/digits.npz",
    "data/splits.json"
]
declared = {p: (ROOT / p).stat().st_size for p in REGISTERED if (ROOT / p).is_file()}
missing = [p for p in REGISTERED if p not in declared]
print("declared source files:", len(REGISTERED))
print("present             :", len(declared))
print("missing             :", missing)
declared

## 4. Inspect the real input data (input audit)

In [ ]:
# --------------------------------------------------------------------------
# Inspect the ACTUAL registered input this lab consumes, through the existing
# course decoder. This is the same file the native programs read.
# --------------------------------------------------------------------------
from shape_lab import data as course_data

for rel in REGISTERED:
    path = ROOT / rel
    if not path.is_file():
        print(f"{rel}: MISSING")
        continue
    print(f"{rel}  ({path.stat().st_size} bytes)")
    if path.suffix == ".csv":
        rows = course_data.read_csv(path) if hasattr(course_data, "read_csv") else None
        if rows is None:
            import csv
            with path.open(newline="", encoding="utf-8-sig") as fh:
                rows = list(csv.reader(fh))
        print("  header :", rows[0][:8])
        print("  data rows:", len(rows) - 1)
    elif path.suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        kind = type(payload).__name__
        size = len(payload) if hasattr(payload, "__len__") else "-"
        print(f"  json {kind} with {size} top-level entries")
    else:
        print("  (binary or image; the native readers decode it in Go and Rust)")

## 5. Resolve the native evidence to inspect (output audit)

In [ ]:
# --------------------------------------------------------------------------
# Resolve the native evidence this notebook will inspect.
#
# TWO EXPLICIT MODES (Phase C):
#   MODE = "fresh"  -> create the evidence NOW, in this notebook's own run
#                      directory, and verify its identity. Nothing older is used.
#   MODE = "review" -> open a specific previous result and label it as such.
#                      It is never silently "the newest folder that looks right".
# --------------------------------------------------------------------------
MODE = "fresh"          # "fresh" or "review"
PINNED_RUN = None       # for MODE="review": an explicit path, e.g. ROOT / "my_work/native_stage01_01"

if MODE not in ("fresh", "review"):
    raise ValueError('MODE must be "fresh" or "review"')

import importlib.util as _ilu
_spec = _ilu.spec_from_file_location("shape_evidence", ROOT / "tools/evidence_pointer.py")
evidence = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(evidence)

NATIVE_RUN = RUN_DIR / "native"

if MODE == "fresh":
    if NATIVE_RUN.exists() and any(NATIVE_RUN.rglob("execution.json")):
        print("This notebook's own run already exists and is reused:")
        print(" ", NATIVE_RUN.relative_to(ROOT))
        print("Delete that directory to force a fresh execution.")
    else:
        print("FRESH EXECUTION: compiling and running the native Go and Rust labs now.")
        print("This creates evidence bound to the CURRENT source and inputs.")
        import shutil
        built = {}
        for _lang in ("go", "rust"):
            _tool = shutil.which("go" if _lang == "go" else "cargo")
            print(f"  {_lang:4s} toolchain:", _tool or "MISSING (see native/guides/SETUP.md)")
        if all(shutil.which(t) for t in ("go", "cargo")):
            # run_labs writes its command logs into this directory, so it must
            # exist first (native_course.py's own CLI does this via new_directory).
            NATIVE_RUN.mkdir(parents=True, exist_ok=True)
            _result = runner.run_labs(("go", "rust"), [LAB_KEY], NATIVE_RUN, root=ROOT)
            print("  run status:", _result["status"])
            print("  identity  :", "recorded" if _result.get("identity") else "MISSING")
        else:
            print("  Cannot execute fresh: install both toolchains, then re-run this cell.")

# In review mode an explicit path is required. Never scan for "something plausible".
if MODE == "review":
    if PINNED_RUN is None:
        print("REVIEW MODE selected but PINNED_RUN is None.")
        print('Set PINNED_RUN = ROOT / "my_work/<the exact run you mean>".')
        found = None
    else:
        found = evidence.resolve(ROOT, LAB_KEY, run_dir=PINNED_RUN, allow_review=False)
        if found is None:
            print("REVIEW MODE: no readable run at", PINNED_RUN)
        else:
            print("REVIEW MODE: opening a PREVIOUS result, not a fresh execution.")
            print("  path:", PINNED_RUN)
            if not found["has_identity"]:
                print("  WARNING: this evidence has no bound identity; it cannot be")
                print("  matched to the current source, binary or inputs.")
else:
    found = evidence.resolve(ROOT, LAB_KEY, run_dir=NATIVE_RUN, allow_review=False)

results = {}
if found is None:
    print()
    print("No usable native evidence for", LAB_KEY)
else:
    results = found["results"]
    run_dir = found["run_dir"]
    print()
    print("run dir      :", run_dir.relative_to(ROOT) if ROOT in run_dir.parents else run_dir)
    print("resolved via :", found["source"])
    print("run status   :", found["status"])
    print("created_at   :", found["created_at"])
    print("languages    :", sorted(results))
    print("has identity :", found["has_identity"])

## 6. Compare Go against Rust, and re-check the input hashes (output audit)

In [ ]:
# --------------------------------------------------------------------------
# Compare the two languages with the runner's own tolerance rules, verify the
# run identity, and check the inputs the native programs consumed.
# --------------------------------------------------------------------------
if not results:
    print("Nothing to compare yet. Create a run and re-run the cell above.")
else:
    if {"go", "rust"} <= set(results):
        comparison = runner.compare_values(results["go"]["metrics"], results["rust"]["metrics"])
        comparison["input_hashes_identical"] = results["go"]["inputs"] == results["rust"]["inputs"]
        print("go == rust (within tolerance):", comparison["passed"])
        print("numeric values compared      :", comparison["numeric_values_compared"])
        print("max absolute difference      :", comparison["max_absolute_difference"])
        print("input hashes identical       :", comparison["input_hashes_identical"])
        if comparison["discrepancies"]:
            print("discrepancies:", comparison["discrepancies"])
        print()
        print("Is this evidence CURRENT source, or stale?")
        for lang in ("go", "rust"):
            recorded = (found["execution"].get("identity") or {}).get("languages", {}).get(lang, {})
            live = runner.source_id(lang, ROOT)
            if not recorded:
                print(f"  {lang}: no identity recorded (legacy evidence) -> cannot confirm")
            else:
                same_src = recorded.get("source_sha256") == live
                print(f"  {lang}: source {'MATCH' if same_src else 'DIFFERS'}; recorded {str(recorded.get('source_sha256'))[:12]} live {live[:12]}")
    else:
        print("Only one language result present:", sorted(results))

    # The native result records the hash of every file it consumed.
    audit = results.get("go", results.get("rust", {})).get("inputs", {})
    for rel in sorted(audit):
        info = audit[rel]
        actual = runner.digest(ROOT / rel)
        flag = "OK " if actual == info["sha256"] else "CHANGED"
        print(f"{flag} {rel}")

## 7. Continue in Go and Rust

Steps 3–6 above audit the **inputs and an existing native run's outputs**; they do
not recompute the lab. The native experiment itself runs in Go and Rust.

| Language | Notebook | What it shows |
|---|---|---|
| Go | `../go/stage04.ipynb` | slice copying, row storage, explicit errors |
| Rust | `../rust/stage04.ipynb` | ownership, borrowing, `Result`, owned working copies |

This notebook is **unassessed**.

In [ ]:
# --------------------------------------------------------------------------
# Now switch language. The SAME lab, the same inputs, the same comparison.
# --------------------------------------------------------------------------
# Read next:
#   native/guides/labs/stage04.md              (the implementation guide)
#   native/guides/LEARNING_PATHS.md                 (the C. Python path)
#   lessons/04_lesson.md
#
# Then open, in this order:
#   ../go/stage04.ipynb
#   ../rust/stage04.ipynb
#
# Three independent challenges for THIS lab:
# 1. Specify the rotation index map by hand.
# 2. Compare a reflection and a rotation for one digit.
# 3. Explain which transformation of a labeled 6 or 9 could invalidate the label.
#
# Boundary to remember:
# A geometric invariance is not necessarily a class-label invariance. General homotopy proofs remain written assignments.
#
# Unassessed. Record your prediction, hand calculation, change, output and one
# limitation in your own notes before moving on.
print("python notebook complete for", LAB_KEY)